# ICU Charting and Staff - Silver Layer

This notebook cleans and prepares the ICU charting and staff data from the Bronze layer.

The Silver layer focuses on improving data quality and consistency while keeping the datasets separate and preserving the relationships between them.

In [0]:
from pyspark.sql import functions as F

# Layer configuration
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# Load Bronze tables
chartevents = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.chartevents_raw")
d_items = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.d_items_raw")
datetimeevents = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.datetimeevents_raw")
caregivers = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.caregivers_raw")
callout = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.callout_raw")

for name, df in {
    "CHARTEVENTS": chartevents,
    "D_ITEMS": d_items,
    "DATETIMEEVENTS": datetimeevents,
    "CAREGIVERS": caregivers,
    "CALLOUT": callout
}.items():
    print(f"{name}: {df.count():,} rows")

## D_ITEMS

`D_ITEMS` provides the lookup information for the `itemid` values used throughout the event data.

The main identifiers are already clean and unique. For Silver, text fields are standardised, blank values are treated as nulls, and `conceptid` is removed because it is completely empty in the demo dataset. Other optional metadata is kept because it still provides useful context when available.

In [0]:
# Clean D_ITEMS text fields
text_columns = [
    "label", "abbreviation", "dbsource", "linksto",
    "category", "unitname", "param_type"
]

d_items_silver = d_items.drop("conceptid")

for column in text_columns:
    d_items_silver = d_items_silver.withColumn(
        column,
        F.when(F.trim(F.col(column)) == "", None)
         .otherwise(F.trim(F.col(column)))
    )

print(f"D_ITEMS Silver: {d_items_silver.count():,} rows")
display(d_items_silver.limit(10))

### D_ITEMS Consistency Check

Before finalising the Silver table, the main categorical fields are checked for simple formatting inconsistencies such as different casing or spacing.

The `itemid` field is already unique, so the main focus here is making sure the supporting metadata is stored consistently.

In [0]:
# Check categorical fields for case/spacing variants
check_columns = ["dbsource", "linksto", "category", "param_type"]

for column in check_columns:
    print(f"\n{column}")

    (
        d_items
        .filter(F.col(column).isNotNull())
        .groupBy(F.lower(F.trim(F.col(column))).alias("standard_value"))
        .agg(
            F.collect_set(F.col(column)).alias("original_values")
        )
        .filter(F.size("original_values") > 1)
        .show(truncate=False)
    )

### D_ITEMS Category Findings

The main categorical fields are already consistent apart from three `category` values that only differ by capitalisation.

These values are standardised in Silver so the same category is not treated as multiple groups later in analytics.

In [0]:
# Standardise known category variants
d_items_silver = (
    d_items_silver
    .withColumn(
        "category",
        F.when(F.lower(F.col("category")) == "drug level", "Drug Level")
         .when(F.lower(F.col("category")) == "abg's", "ABG's")
         .when(F.lower(F.col("category")) == "vbg's", "VBG's")
         .otherwise(F.col("category"))
    )
)

### D_ITEMS Silver Result

`D_ITEMS` has been cleaned while keeping all source records. The completely empty `conceptid` field was removed, text fields were standardised, and meaningful missing values were left unchanged.

In [0]:
# Create Silver schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

# Save cleaned D_ITEMS table
(
    d_items_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.d_items")
)

print(f"Created: {CATALOG}.{SILVER_SCHEMA}.d_items")

## CAREGIVERS

`CAREGIVERS` provides the lookup information for staff IDs used in the ICU event tables.

The caregiver identifier is already complete and unique. For Silver, the main cleaning is limited to standardising the text fields while keeping missing descriptive information as null.

In [0]:
# Clean CAREGIVERS text fields
caregivers_silver = caregivers

for column in ["label", "description"]:
    caregivers_silver = caregivers_silver.withColumn(
        column,
        F.when(F.trim(F.col(column)) == "", None)
         .otherwise(F.trim(F.col(column)))
    )

print(f"CAREGIVERS Silver: {caregivers_silver.count():,} rows")
display(caregivers_silver.limit(10))

### Label Check

Before saving the Silver table, the relationship between `label` and `description` is checked to make sure the same staff label is not being used with different descriptions.

In [0]:
# Check whether each label maps to more than one description
label_check = (
    caregivers
    .groupBy("label")
    .agg(
        F.countDistinct("description").alias("description_count")
    )
    .filter(F.col("description_count") > 1)
)

display(label_check)

### CAREGIVERS Label Findings

- Several caregiver labels map to multiple descriptions, so `label` and `description` are not a one-to-one relationship.
- Some labels also appear with different capitalisation, such as `RPH` / `RPh` and `MDs` / `MDS`, which may be simple formatting differences.
- Both fields should be kept in the Silver table for now rather than combining them or assuming the label fully explains the caregiver role.

In [0]:
# Inspect labels with multiple descriptions
(
    caregivers
    .filter(F.col("label").isin(
        [row["label"] for row in label_check.collect()]
    ))
    .select("label", "description")
    .distinct()
    .orderBy("label", "description")
    .show(100, truncate=False)
)

### CAREGIVERS Cleaning Decision

The `label` and `description` fields do not have a one-to-one relationship, so they should remain as separate fields rather than being combined or used to overwrite one another.

Some labels only differ by capitalisation, so the Silver layer will standardise label casing for consistency. Missing descriptions and different description values will be preserved because they may represent genuine differences in how caregiver records were captured.

In [0]:
# Clean CAREGIVERS fields
caregivers_silver = (
    caregivers
    .withColumn(
        "label",
        F.when(F.trim(F.col("label")) == "", None)
         .otherwise(F.upper(F.trim(F.col("label"))))
    )
    .withColumn(
        "description",
        F.when(F.trim(F.col("description")) == "", None)
         .otherwise(F.trim(F.col("description")))
    )
)

### CAREGIVERS Silver Result

`CAREGIVERS` has been cleaned while keeping all source records. Staff labels have been standardised for consistent casing, while the original description values and meaningful nulls have been preserved.

In [0]:
# Save cleaned CAREGIVERS table
(
    caregivers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.caregivers")
)

print(f"Created: {CATALOG}.{SILVER_SCHEMA}.caregivers")

## CHARTEVENTS

`CHARTEVENTS` contains the main ICU observations recorded for patients.

The Silver layer keeps the different value formats and source-specific fields because their missing values are often expected. Basic text cleaning is applied and `dbsource` is added from `D_ITEMS` so CareVue and MetaVision records can be clearly distinguished.

In [0]:
# Add source system and clean text fields
chartevents_silver = (
    chartevents
    .join(
        d_items_silver.select("itemid", "dbsource"),
        on="itemid",
        how="left"
    )
)

text_columns = ["value", "valueuom", "resultstatus", "stopped"]

for column in text_columns:
    chartevents_silver = chartevents_silver.withColumn(
        column,
        F.when(F.trim(F.col(column)) == "", None)
         .otherwise(F.trim(F.col(column)))
    )

print(f"CHARTEVENTS Silver: {chartevents_silver.count():,} rows")
display(chartevents_silver.limit(10))

### CHARTEVENTS Silver Result

`CHARTEVENTS` has been cleaned while keeping the different chart value formats and source-specific fields intact. The source system has also been added so CareVue and MetaVision records can be interpreted correctly later.

In [0]:
# Save cleaned CHARTEVENTS table
(
    chartevents_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.chartevents")
)

print(f"Created: {CATALOG}.{SILVER_SCHEMA}.chartevents")

## DATETIMEEVENTS

`DATETIMEEVENTS` stores ICU events where the recorded value is a date or time.

The Silver layer keeps the original event structure, preserves source-specific fields and adds `dbsource` from `D_ITEMS` so CareVue and MetaVision records can be distinguished.

In [0]:
# Add source system and clean text fields
datetimeevents_silver = (
    datetimeevents
    .join(
        d_items_silver.select("itemid", "dbsource"),
        on="itemid",
        how="left"
    )
)

for column in ["valueuom", "resultstatus", "stopped"]:
    datetimeevents_silver = datetimeevents_silver.withColumn(
        column,
        F.when(F.trim(F.col(column)) == "", None)
         .otherwise(F.trim(F.col(column)))
    )

print(f"DATETIMEEVENTS Silver: {datetimeevents_silver.count():,} rows")
display(datetimeevents_silver.limit(10))

### DATETIMEEVENTS Silver Result

`DATETIMEEVENTS` has been prepared for Silver while keeping the original date and time values intact. The source system has been added so CareVue and MetaVision records can be interpreted correctly later.

In [0]:
# Save cleaned DATETIMEEVENTS table
(
    datetimeevents_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.datetimeevents")
)

print(f"Created: {CATALOG}.{SILVER_SCHEMA}.datetimeevents")

## CALLOUT

`CALLOUT` contains information about patient callout requests, including care units, request details, outcomes and related timestamps.

The Silver layer keeps the full callout structure while standardising text fields and preserving optional values that may only apply to certain requests.

In [0]:
# Clean CALLOUT text fields
callout_silver = callout

text_columns = [
    "submit_careunit",
    "curr_careunit",
    "callout_service",
    "callout_status",
    "callout_outcome",
    "acknowledge_status"
]

for column in text_columns:
    callout_silver = callout_silver.withColumn(
        column,
        F.when(F.trim(F.col(column)) == "", None)
         .otherwise(F.trim(F.col(column)))
    )

print(f"CALLOUT Silver: {callout_silver.count():,} rows")
display(callout_silver.limit(10))

### CALLOUT Silver Result

`CALLOUT` has been cleaned while keeping the full callout structure and all source records. Text fields have been standardised, while optional care-unit, reservation and timing fields have been preserved as null where they are not available.

In [0]:
# Save cleaned CALLOUT table
(
    callout_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.callout")
)

print(f"Created: {CATALOG}.{SILVER_SCHEMA}.callout")

## Silver Validation

The completed Silver tables are checked against their Bronze sources to confirm that the cleaning process has preserved all records.

This provides a final check before the Silver pipeline is committed to the project repository.


In [0]:
# Validate Bronze and Silver row counts
table_pairs = {
    "chartevents_raw": "chartevents",
    "d_items_raw": "d_items",
    "datetimeevents_raw": "datetimeevents",
    "caregivers_raw": "caregivers",
    "callout_raw": "callout"
}

for bronze_table, silver_table in table_pairs.items():
    bronze_count = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{bronze_table}"
    ).count()

    silver_count = spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.{silver_table}"
    ).count()

    print(
        f"{silver_table}: "
        f"bronze={bronze_count:,} | silver={silver_count:,} | "
        f"match={bronze_count == silver_count}"
    )

### Silver Layer Result

All five Silver tables were created successfully and retained the same number of records as their Bronze sources.

The Silver layer now applies targeted cleaning and standardisation while preserving meaningful null values, source-system differences and the relationships between the ICU charting and staff datasets.